In [ ]:
sys.path.append('..')
import os
from torch import nn
from random import shuffle
from transformers import AutoTokenizer, AutoModel, EsmModel, EsmTokenizer
import torch
import glob
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from nn_data_prep import TFDataset
from model_pro4 import AttachedModel,create_attached_model
from model_wrapper_pro import ModelWrapper
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, f1_score, accuracy_score

In [ ]:
test_data=torch.load("./test/test.pt",weights_only=False)
model_wrapper = ModelWrapper(
    protein_model_name="facebook/esm2_t6_8M_UR50D",
    dna_encoding_method='transformer',
    dna_max_length=40,
    namefile='permu_shuffle',
    protein_max_length=903,
    save_path='./savefile'
)
model_wrapper.get_model_settings()
model_wrapper.load_model('./model.pt') # model weight downloaded at ""

In [ ]:
model_wrapper.test(test_data)

In [ ]:
class DataFrameDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return row['tf_id'], row['tf_seq'], row['sequence']

test_data_table=read_csv('test.csv',index_col=0)
test_data_table = DataFrameDataset(test_data_table)
test_loader=torch.utils.data.DataLoader(test_data_table, batch_size=16, shuffle=False)
test_data_table['predict']=model_wrapper.simple_predict(test_loader)